# ANER — Experiment 2: ML + rules

**Question:** Can bounded regex and name rules recover missed entities and improve span boundaries without losing too many correct ML predictions?

We reuse the exact checkpoint, approved annotations (guidelines 2.1), and 50% training / 25% evaluation / 25% test partition from experiment 1. This experiment applies rules during prediction; it does not retrain the model or modify gold annotations.

Compare **ML baseline**, **rules only**, and **ML + rules**, using the same exact-span scoring. A match requires identical character start, end and label. Review precision, recall, F1 and support per label, together with newly introduced errors. A secondary comparison counts each whitespace/case-normalized evaluation text once, reducing the weight of repeated boilerplate while preserving each selected passage and its original gold offsets.

The initial rules cover:

| Label | Rule scope |
|---|---|
| ORG | PLACSP, its full platform name with an attached acronym, Registro Mercantil, ROLECE; Seguridad Social only with a narrow institutional context such as “con la” or “ante la” |
| LOC | A small gazetteer initially containing Tenerife; abstain when preceded by an institution name |
| DATE | Valid Spanish dates with or without a year, and numeric dates; no standalone-year rule in this version |
| LAW | LCSP references and attached articles, articles explicitly linked to a numbered Reglamento, and directive identifiers |
| SCHEDULE | Clock times, weekday time windows and the supported abbreviated L–J / V form |
| EMAIL | Complete email addresses, excluding surrounding punctuation |
| MISC | Technical standard identifiers, Código CPV identifiers with an attached quoted description, and Certificado ROLECE references |

PER and EVENT receive no new rules. Existing ML predictions remain unless a selected rule overlaps them or an explicitly handled generic subject use of Seguridad Social is removed. Prioritize complete references and prohibit overlapping entities. Keep internal punctuation and attached legal articles; exclude sentence punctuation. A complete ML LAW citation protects dates inside it. Rule spans must align strictly with the saved tokenizer: unaligned matches are skipped and logged.

These are intentionally limited rules, not full coverage of guidelines 2.1. The rules were designed after reviewing evaluation errors, so evaluation gains measure development progress. The existing test set was also previously inspected: any later comparison on it is exploratory. An independent final estimate requires fresh, untouched data.

## Start here

Download `ANER_Experiment_2.py` alongside this notebook. In your experiment-1 Colab notebook, run `export_run()` to download **ANER_Run_Outputs.zip**, which includes your trained checkpoint and split files. Open this notebook in Colab and upload those two files in the setup cell below. No training is required.

If the experiment-1 variables are still available in the current runtime, the setup cell reuses them and asks only for the Python file. Do not substitute `ANER_Colab_Project.zip`: that input bundle does not contain your trained model.

In [1]:
%pip -q install "spacy==3.8.16" pandas

In [2]:
from collections import Counter
from pathlib import Path
import hashlib
import importlib
import json
import zipfile

import pandas as pd
import spacy
from IPython.display import display
from google.colab import files

RUN_HYBRID_TEST = False
required_baseline = ['trained_model', 'split_rows', 'split_indices', 'sources',
                     'RUN_DIR', 'run_metadata', 'SPLIT_SHA256', 'APPROVED_SHA256', 'LABELS']
reuse_runtime = all(globals().get(name) is not None for name in required_baseline)
print('Upload ANER_Experiment_2.py' + ('.' if reuse_runtime else ' and ANER_Run_Outputs.zip.'))
uploaded = files.upload()
if 'ANER_Experiment_2.py' not in uploaded:
    raise RuntimeError('Upload the accompanying ANER_Experiment_2.py file.')
script_path = Path.cwd() / 'ANER_Experiment_2.py'
script_path.write_bytes(uploaded['ANER_Experiment_2.py'])
_ANER_EXPERIMENT_2_SOURCE = str(script_path)
importlib.invalidate_caches()
import ANER_Experiment_2 as experiment_2
experiment_2 = importlib.reload(experiment_2)
experiment_2.RUN_HYBRID_TEST = False

if not reuse_runtime:
    if 'ANER_Run_Outputs.zip' not in uploaded:
        raise RuntimeError('Use export_run() in experiment 1, then upload ANER_Run_Outputs.zip.')
    input_root = Path('/content/ANER_Experiment_2/inputs')
    input_root.mkdir(parents=True, exist_ok=True)
    archive_path = input_root / 'ANER_Run_Outputs.zip'
    archive_path.write_bytes(uploaded['ANER_Run_Outputs.zip'])
    # Reject archive paths outside the designated folder.
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (input_root / member.filename).resolve()
            if not target.is_relative_to(input_root.resolve()) or (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Archive contains an unsupported path: ' + member.filename)
        archive.extractall(input_root)
    candidates = [p for p in input_root.glob('models/*/run_metadata.json')
                  if (p.parent / 'model-best').is_dir()]
    if len(candidates) != 1:
        raise RuntimeError('Expected exactly one baseline checkpoint in the run export; found ' + str(len(candidates)))
    RUN_DIR = candidates[0].parent
    run_metadata = json.loads(candidates[0].read_text(encoding='utf-8'))
    data_root = input_root / 'data' / 'processed'
    manifest_path = data_root / 'split_manifest.json'
    SPLIT_SHA256 = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    APPROVED_SHA256 = run_metadata['annotation_sha256']
    assert manifest['annotation_sha256'] == APPROVED_SHA256
    assert run_metadata['split_manifest_sha256'] == SPLIT_SHA256
    data = {name: json.loads((data_root / (name + '.json')).read_text(encoding='utf-8'))
            for name in ['train', 'evaluation', 'test']}
    LABELS = data['train']['classes']
    split_rows, split_indices, sources = {}, {}, {}
    for name, item in data.items():
        meta = item['metadata']
        assert item['classes'] == LABELS
        assert meta['annotation_sha256'] == APPROVED_SHA256
        assert meta['split_manifest_sha256'] == SPLIT_SHA256
        indices = meta['original_annotation_indices']
        assert indices == manifest['original_annotation_indices'][name]
        assert len(indices) == len(item['annotations']) == len(meta['record_sources'])
        split_rows[name] = item['annotations']
        split_indices[name] = indices
        sources.update(dict(zip(indices, meta['record_sources'])))
    trained_model = spacy.load(RUN_DIR / 'model-best')
    for name, rows in split_rows.items():
        for text, annotation in rows:
            doc = trained_model.make_doc(text)
            for start, end, label in annotation['entities']:
                assert label in LABELS
                assert doc.char_span(start, end, label=label, alignment_mode='strict') is not None
    print('Loaded saved baseline:', RUN_DIR / 'model-best')
else:
    print('Reusing baseline:', RUN_DIR / 'model-best')
print('Record counts:', {name: len(rows) for name, rows in split_rows.items()})

Upload ANER_Experiment_2.py and ANER_Run_Outputs.zip.


Saving ANER_Experiment_2.py to ANER_Experiment_2.py
Saving ANER_Run_Outputs.zip to ANER_Run_Outputs.zip
Loaded saved baseline: /content/ANER_Experiment_2/inputs/models/20261007T134812850576Z/model-best
Record counts: {'train': 582, 'evaluation': 291, 'test': 292}


In [3]:
def score_rows(model, rows, include_predictions=False):
    """Strict entity scoring: start, end and label must all match exactly."""
    tp, fp, fn, support = Counter(), Counter(), Counter(), Counter()
    predictions = []
    for doc, (text, annotation) in zip(model.pipe((row[0] for row in rows), batch_size=32), rows):
        gold = {tuple(entity) for entity in annotation['entities']}
        predicted = {(e.start_char, e.end_char, e.label_) for e in doc.ents}
        tp.update(entity[2] for entity in gold & predicted)
        fp.update(entity[2] for entity in predicted - gold)
        fn.update(entity[2] for entity in gold - predicted)
        support.update(entity[2] for entity in gold)
        if include_predictions:
            predictions.append({'text': text, 'gold': sorted(gold), 'predicted': sorted(predicted),
                                'false_positives': sorted(predicted - gold), 'false_negatives': sorted(gold - predicted)})
    def metrics(t, f, n, gold):
        return {'precision': t / (t + f) if t + f else None,
                'recall': t / (t + n) if t + n else None,
                'f1': 2 * t / (2 * t + f + n) if 2 * t + f + n else None,
                'tp': t, 'fp': f, 'fn': n, 'gold_support': gold}
    micro = metrics(sum(tp.values()), sum(fp.values()), sum(fn.values()), sum(support.values()))
    per_label = {label: metrics(tp[label], fp[label], fn[label], support[label]) for label in LABELS}
    result = {'records': len(rows), 'micro': micro, 'per_label': per_label}
    if include_predictions:
        result['predictions'] = predictions
    return result


## Evaluate the three systems

The cell below runs evaluation only and saves the full rule configuration, predictions, label scores and changes. The original ML pipeline stays unchanged. “Rules only” helps show how much of any improvement comes from recognizing these supported patterns rather than the learned model.

The configuration and rule-source hashes are saved with baseline metadata. A saved hybrid pipeline is reloaded and its predictions checked against the evaluated pipeline. Undefined metrics remain `None` / `NaN`; they do not mean successful recognition.

In [4]:
experiment_2.RUN_HYBRID_TEST = False
globals().update(experiment_2.run_experiment_2(globals()))

EVALUATION — same checkpoint, same records, exact-span scoring


,precision,recall,f1,tp,fp,fn,gold_support
system,,,,,,,
ML baseline,0.632479,0.493333,0.554307,74,43,76,150
Rules only,1.000000,0.220000,0.360656,33,0,117,150
ML + rules,0.748148,0.673333,0.708772,101,34,49,150


,,precision,recall,f1,gold_support
label,system,,,,
ORG,ML baseline,0.000000,0.000000,0.000000,30
PER,ML baseline,NaN,NaN,NaN,0
LOC,ML baseline,0.500000,0.250000,0.333333,4
DATE,ML baseline,0.666667,0.400000,0.500000,5
MISC,ML baseline,0.647619,0.666667,0.657005,102
LAW,ML baseline,0.750000,0.500000,0.600000,6
EMAIL,ML baseline,NaN,0.000000,0.000000,1
SCHEDULE,ML baseline,NaN,0.000000,0.000000,2
EVENT,ML baseline,0.000000,NaN,0.000000,0


NORMALIZED UNIQUE EVALUATION TEXTS: 166


,precision,recall,f1,tp,fp,fn,gold_support
system,,,,,,,
ML baseline,0.633333,0.471074,0.540284,57,33,64,121
Rules only,1.000000,0.239669,0.386667,29,0,92,121
ML + rules,0.771429,0.669421,0.716814,81,24,40,121


Changed evaluation passages: 17
Recovered gold: 27 | lost previously correct gold: 0
Introduced false positives: 0 | removed false positives: 9
Test scoring is disabled. Review evaluation changes before enabling RUN_HYBRID_TEST.
Saved comparison, rule configuration, changes and hybrid model: /content/ANER_Experiment_2/inputs/models/20261007T134812850576Z/experiment_2/20261007T150433193532Z


## Review improvements and regressions

Inspect both recovered gold entities and newly introduced errors before changing rules. `show_rule_change(0)` displays a complete passage and the rule decisions; change the row number to review others. The original annotation index and source line remain available.

Use evaluation data for rule development. Broad ORG patterns can mislabel subject areas; short LAW matches can cut off full citations; place names can sit inside institutional names. Preserve these scope decisions when extending the rule set.

In [5]:
change_table = pd.DataFrame([
    {'review_row': number, 'original_annotation_index': row['original_annotation_index'],
     'source_file': row['source'].get('source_file'), 'line_number': row['source'].get('line_number'),
     'recovered_gold': len(row['recovered_gold']), 'lost_gold': len(row['lost_gold']),
     'removed_false_positives': len(row['removed_false_positives']),
     'introduced_false_positives': len(row['introduced_false_positives'])}
    for number, row in enumerate(hybrid_rule_changes)
])
display(change_table.head(30))

def show_rule_change(review_row):
    if not 0 <= review_row < len(hybrid_rule_changes):
        raise IndexError('Choose a row from change_table.')
    row = hybrid_rule_changes[review_row]
    print('Original index:', row['original_annotation_index'], '| source:', row['source'])
    print(row['text'])
    for key in ['recovered_gold', 'lost_gold', 'removed_false_positives', 'introduced_false_positives']:
        print(key, [(start, end, label, row['text'][start:end]) for start, end, label in row[key]])
    display(pd.DataFrame(row['rule_audit']))

if hybrid_rule_changes:
    show_rule_change(0)

,review_row,original_annotation_index,source_file,line_number,recovered_gold,lost_gold,removed_false_positives,introduced_false_positives
0,0,124,raw_data_sample2.txt,134,1,0,0,0
1,1,145,raw_data_sample2.txt,156,3,0,0,0
2,2,179,raw_data_sample2.txt,190,2,0,1,0
3,3,539,raw_data_sample3.txt,285,1,0,0,0
4,4,540,raw_data_sample3.txt,286,3,0,3,0
5,5,541,raw_data_sample3.txt,287,1,0,0,0
6,6,542,raw_data_sample3.txt,288,1,0,0,0
7,7,624,raw_data_sample4.txt,13,1,0,0,0
8,8,819,raw_data_sample4.txt,215,1,0,1,0
9,9,825,raw_data_sample4.txt,221,3,0,0,0


Original index: 124 | source: {'source_file': 'raw_data_sample2.txt', 'line_number': 134, 'original_source_start': 10080, 'original_source_end_exclusive': 10703, 'previous_annotation_index': 126, 'retained_source_start': 10080, 'retained_source_end_exclusive': 10703}
Ver PCAP. Se presentará el Anexo II del PCAP debidamente firmado por el representante legal de la empresa licitante en formato adobe generado con la plantilla que se anexa. El documento se presentará a través de la herramienta de preparación y presentación de ofertas de la PLACSP Relación de obras, servicios o suministros según el caso similares (incluida en el anexo II) relativos al mismo código CPV ejecutados en los años señalados en los Pliegos cuyo importe anual acumulado en el año de mayor ejecución no sea menos de la cantidad reflejada en los Pliegos en el que se indique la fecha, el importe y el destinatario

recovered_gold [(274, 280, 'ORG', 'PLACSP')]
lost_gold []
removed_false_positives []
introduced_false_positi

,rule_id,start,end,label,text,priority,decision
0,organization:PLACSP,274,280,ORG,PLACSP,80,accepted


## Findings to complete after running

- Compare evaluation precision, recall and F1 for each system, especially ORG, DATE, LOC, LAW and SCHEDULE. Include gold support.
- Record recovered entities, lost correct predictions and new false positives. Check whether MISC performance falls as other labels improve.
- Separate name-dictionary coverage from structural improvements to date, schedule and citation boundaries.
- Compare all-record and normalized unique-text scores to see how much repeated text affects the result.
- Report that evaluation errors informed rule design. Do not claim improved generalization from a development-set gain alone.

Keep test scoring disabled while revising rules. If you later choose to score the existing test set, describe it as an exploratory comparison because it has already been inspected.

For GitHub, commit this notebook, `ANER_Experiment_2.py` and the generated comparison / audit / metadata JSON files. Keep the baseline results alongside them and avoid committing model weights unnecessarily.

Implementation reference: [spaCy rule-based matching](https://spacy.io/usage/rule-based-matching) and [custom pipeline components](https://spacy.io/usage/processing-pipelines#custom-components).

In [6]:
output_zip = HYBRID_RUN_DIR.parent / (HYBRID_RUN_DIR.name + '_experiment_2.zip')
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(HYBRID_RUN_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(HYBRID_RUN_DIR))
    archive.write(script_path, 'ANER_Experiment_2.py')
print('Download includes comparison reports, audit, metadata and the saved hybrid model.')
files.download(str(output_zip))

Download includes comparison reports, audit, metadata and the saved hybrid model.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Experiment 2 — Findings**

Adding regular expressions and name-based rules improved the system without retraining the ML model. Evaluation F1 increased from **55.43% to 70.88%**. The rules recovered **27 missed entities** and removed **9 incorrect predictions**, without introducing new errors or removing correct predictions on this evaluation set.

The improvement remained when repeated texts were counted only once: F1 increased from **54.03% to 71.68%**. This suggests that repetition alone does not explain the gain.

The rules helped recognize organization names, dates, legal references, email addresses and schedules, while correcting some entity boundaries. Most remaining errors concern ORG and MISC. Several labels achieved perfect evaluation scores, but they had very few examples, so these results should be interpreted cautiously.

This experiment shows that targeted rules can complement an ML model. However, the rules were developed using evaluation errors, so these scores describe progress on development data rather than independently verified performance on new texts.